In [1]:
import requests
import time
import pandas as pd
import os
import re
import numpy as np

In [2]:
raw_trait = pd.read_csv(os.path.join(os.getcwd(), "pgs_traits_data.csv"))

In [3]:
contain_catagories = ["Cancer", "Cardiovascular disease", "Digestive system disorder4", "Immune system disorder", "Metabolic disorder", "Neurological disorder", "Other disease"]
raw_trait = raw_trait[raw_trait["Trait Category"].isin(contain_catagories)]

In [4]:
print(raw_trait.shape, raw_trait.head(5))

(260, 4)           Trait (ontology term label)  \
0           Abdominal Aortic Aneurysm   
3                        Abnormal EKG   
6  ACPA-negative rheumatoid arthritis   
7  ACPA-positive rheumatoid arthritis   
8                 acute kidney injury   

                      Trait Identifier (ontology ID)          Trait Category  \
0  EFO_0004214: http://www.ebi.ac.uk/efo/EFO_0004214  Cardiovascular disease   
3  HP_0003115: http://purl.obolibrary.org/obo/HP_...  Cardiovascular disease   
6  EFO_0009460: http://www.ebi.ac.uk/efo/EFO_0009460  Immune system disorder   
7  EFO_0009459: http://www.ebi.ac.uk/efo/EFO_0009459  Immune system disorder   
8  MONDO_0002492: http://purl.obolibrary.org/obo/...           Other disease   

   Number of Related PGS  
0                      6  
3                      1  
6                      1  
7                      1  
8                      2  


In [6]:
aou_gene_data = pd.read_csv("D:\Biostat_Research\Genetic_Agent\ICD10CM_childrencode_WGS.csv")

In [26]:
import re
results = []
for idx, row in raw_trait.iterrows():
    ontology = str(row["Trait (ontology term label)"]).lower()
    words = re.findall(r'\b\w+\b', ontology)
    
    matched = aou_gene_data[
        aou_gene_data["description (HIPAA-covered)"]
        .apply(lambda x: all(word in str(x).lower() for word in words))
    ]
    
    if not matched.empty:
        first_match = matched.iloc[0]
        icd = first_match["ICD10_code"]
        description = first_match["description (HIPAA-covered)"]
    else:
        icd = np.nan
        description = np.nan
    
    results.append({
        "ontology": ontology,
        "pgs_num": row["Number of Related PGS"],
        "icd": icd,
        "description": description
    })

result_df = pd.DataFrame(results)

In [27]:
print(result_df.shape)

(260, 4)


In [28]:
print(result_df)

                               ontology  pgs_num    icd  \
0             abdominal aortic aneurysm        6   I714   
1                          abnormal ekg        1  R9431   
2    acpa-negative rheumatoid arthritis        1    NaN   
3    acpa-positive rheumatoid arthritis        1    NaN   
4                   acute kidney injury        2    NaN   
..                                  ...      ...    ...   
255                  vasomotor rhinitis        1   J300   
256              venous thromboembolism       11    NaN   
257           vertebral column disorder        1    NaN   
258              vitamin b12 deficiency        1   D519   
259                            vitiligo        3    L80   

                                           description  
0    0 Abdominal aortic aneurysm, without rupture  ...  
1    Abnormal electrocardiogram [ECG] [EKG]        ...  
2                                                  NaN  
3                                                  NaN  
4     

In [ ]:
result_df.to_csv(os.path.join(os.getcwd(),"trait_list_260218.csv"), index=False)